# Standard RAG vs GraphRAG — Comparison & Visualizations

This notebook visualizes the performance differences between:
- **Standard RAG** — chunk → embed → cosine similarity → LLM
- **GraphRAG** — knowledge graph + community summaries → LLM

**Prerequisites:**
1. Run `python examples/06_benchmark.py` to generate `output/benchmark_results.json`
2. (Optional) Run `python -m graphrag index --root ragtest` first for GraphRAG results

In [ ]:
import json
import sys
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

RESULTS = ROOT / 'output' / 'benchmark_results.json'

if not RESULTS.exists():
    print(f'Results file not found: {RESULTS}')
    print('Run: python examples/06_benchmark.py')
else:
    data = json.loads(RESULTS.read_text('utf-8'))
    print(f'Loaded results: {len(data["standard_rag"])} RAG + {len(data["graphrag"])} GraphRAG answers')

## 1. Build comparison dataframe

In [ ]:
rag_df = pd.DataFrame(data['standard_rag'])
grag_df = pd.DataFrame(data['graphrag'])

# Merge on question id
merged = rag_df[['id','type','question','answer','latency_s','prompt_tokens','completion_tokens']].copy()
merged.columns = ['id','type','question','rag_answer','rag_latency','rag_prompt_tok','rag_completion_tok']

if not grag_df.empty:
    grag_sub = grag_df[['id','answer','latency_s','method']].copy()
    grag_sub.columns = ['id','graphrag_answer','graphrag_latency','graphrag_method']
    merged = merged.merge(grag_sub, on='id', how='left')
else:
    merged['graphrag_answer'] = '(not run)'
    merged['graphrag_latency'] = np.nan
    merged['graphrag_method'] = np.nan

merged['rag_answer_len'] = merged['rag_answer'].str.len()
merged['graphrag_answer_len'] = merged['graphrag_answer'].str.len()

merged[['id','type','rag_answer_len','graphrag_answer_len','rag_latency','graphrag_latency']]

## 2. Answer length by question type

GraphRAG tends to produce longer, more synthesized answers for thematic questions.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Left: answer length per question ──────────────────────────────────────────
ax = axes[0]
x = np.arange(len(merged))
w = 0.38
bars1 = ax.bar(x - w/2, merged['rag_answer_len'], w, label='Standard RAG', color='#4C72B0')
bars2 = ax.bar(x + w/2, merged['graphrag_answer_len'], w, label='GraphRAG', color='#DD8452')
ax.set_xticks(x)
ax.set_xticklabels(merged['id'], fontsize=9)
ax.set_ylabel('Answer length (chars)')
ax.set_title('Answer Length per Question')
ax.legend()

# Color x-tick labels by question type
type_colors = {'factual': '#2ecc71', 'relational': '#3498db', 'thematic': '#9b59b6', 'multi_hop': '#e74c3c'}
for tick, qtype in zip(ax.get_xticklabels(), merged['type']):
    tick.set_color(type_colors.get(qtype, 'black'))

# ── Right: average length by type ─────────────────────────────────────────────
ax2 = axes[1]
type_order = ['factual', 'relational', 'multi_hop', 'thematic']
rag_by_type    = merged.groupby('type')['rag_answer_len'].mean().reindex(type_order)
graphrag_by_type = merged.groupby('type')['graphrag_answer_len'].mean().reindex(type_order)

x2 = np.arange(len(type_order))
ax2.bar(x2 - w/2, rag_by_type,    w, label='Standard RAG', color='#4C72B0')
ax2.bar(x2 + w/2, graphrag_by_type, w, label='GraphRAG',   color='#DD8452')
ax2.set_xticks(x2)
ax2.set_xticklabels(type_order)
ax2.set_ylabel('Avg answer length (chars)')
ax2.set_title('Average Answer Length by Question Type')
ax2.legend()

# Legend for question type colors
handles = [mpatches.Patch(color=c, label=t) for t, c in type_colors.items()]
axes[0].legend(handles=handles + [mpatches.Patch(color='#4C72B0', label='Standard RAG'),
                                    mpatches.Patch(color='#DD8452', label='GraphRAG')])
plt.tight_layout()
plt.savefig(ROOT / 'output' / 'viz_answer_length.png', bbox_inches='tight')
plt.show()

## 3. Entity coverage heatmap

For each question we defined `expected_entities` — key terms we'd hope to see in a good answer. This heatmap shows how many each system covers.

In [ ]:
def entity_coverage(answer: str, entities: list[str]) -> float:
    """Fraction of expected entities mentioned in the answer (case-insensitive)."""
    if not entities:
        return np.nan
    ans_lower = answer.lower()
    hits = sum(1 for e in entities if e.lower() in ans_lower)
    return hits / len(entities)

questions_meta = {q['id']: q for q in data['questions']}

coverage_data = []
for _, row in merged.iterrows():
    entities = questions_meta[row['id']].get('expected_entities', [])
    if not entities:
        continue
    coverage_data.append({
        'id': row['id'],
        'type': row['type'],
        'Standard RAG': entity_coverage(row['rag_answer'], entities),
        'GraphRAG': entity_coverage(str(row['graphrag_answer']), entities),
    })

cov_df = pd.DataFrame(coverage_data).set_index('id')

fig, ax = plt.subplots(figsize=(8, 5))
heat_data = cov_df[['Standard RAG', 'GraphRAG']]
sns.heatmap(
    heat_data, annot=True, fmt='.0%', cmap='RdYlGn',
    vmin=0, vmax=1, linewidths=0.5, ax=ax,
    cbar_kws={'label': 'Entity coverage'}
)
ax.set_title('Expected Entity Coverage in Answers\n(questions with expected entities only)')
ax.set_ylabel('')
plt.tight_layout()
plt.savefig(ROOT / 'output' / 'viz_entity_coverage.png', bbox_inches='tight')
plt.show()
print('\nMean entity coverage:')
print(heat_data.mean().to_string())

## 4. Latency comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Per-question latency
ax = axes[0]
ax.plot(merged['id'], merged['rag_latency'],    'o-', label='Standard RAG', color='#4C72B0')
ax.plot(merged['id'], merged['graphrag_latency'], 's-', label='GraphRAG',   color='#DD8452')
ax.set_ylabel('Latency (seconds)')
ax.set_title('Query Latency per Question')
ax.tick_params(axis='x', rotation=45)
ax.legend()

# Average latency by type
ax2 = axes[1]
type_order = ['factual', 'relational', 'multi_hop', 'thematic']
rag_lat = merged.groupby('type')['rag_latency'].mean().reindex(type_order)
grag_lat = merged.groupby('type')['graphrag_latency'].mean().reindex(type_order)

x = np.arange(len(type_order))
w = 0.35
ax2.bar(x - w/2, rag_lat,  w, label='Standard RAG', color='#4C72B0')
ax2.bar(x + w/2, grag_lat, w, label='GraphRAG',     color='#DD8452')
ax2.set_xticks(x)
ax2.set_xticklabels(type_order)
ax2.set_ylabel('Avg latency (seconds)')
ax2.set_title('Average Latency by Question Type')
ax2.legend()

plt.tight_layout()
plt.savefig(ROOT / 'output' / 'viz_latency.png', bbox_inches='tight')
plt.show()

## 5. Side-by-side answer comparison

In [ ]:
from IPython.display import display, HTML

def compare_answers(qid: str) -> None:
    row = merged[merged['id'] == qid].iloc[0]
    qmeta = questions_meta[qid]

    html = f"""
    <div style='font-family:sans-serif; max-width:1200px'>
      <h3 style='color:#333'>[{qid}] {row['question']}</h3>
      <p><em>Type: <b>{row['type']}</b></em></p>
      <table style='width:100%; border-collapse:collapse'>
        <tr>
          <th style='background:#4C72B0;color:white;padding:8px;width:50%;text-align:left'>Standard RAG ({row['rag_latency']}s)</th>
          <th style='background:#DD8452;color:white;padding:8px;width:50%;text-align:left'>GraphRAG ({row.get('graphrag_latency','?')}s | {row.get('graphrag_method','')})</th>
        </tr>
        <tr>
          <td style='padding:10px;vertical-align:top;background:#f0f4ff;border:1px solid #ddd;white-space:pre-wrap'>{row['rag_answer']}</td>
          <td style='padding:10px;vertical-align:top;background:#fff8f0;border:1px solid #ddd;white-space:pre-wrap'>{row['graphrag_answer']}</td>
        </tr>
      </table>
    </div>
    """
    display(HTML(html))

# Show all questions
for qid in merged['id']:
    compare_answers(qid)
    print()

## 6. Knowledge graph visualization

Visualize the GraphRAG knowledge graph built from the sample documents.

In [ ]:
import networkx as nx

OUTPUT_DIR = ROOT / 'ragtest' / 'output'

def load_parquet(keyword):
    files = sorted(OUTPUT_DIR.rglob(f'*{keyword}*.parquet'), key=lambda p: p.stat().st_mtime)
    if not files:
        return pd.DataFrame()
    return pd.read_parquet(files[-1])

entities = load_parquet('entities')
rels = load_parquet('relationships')

print(f'Entities: {len(entities)}, Relationships: {len(rels)}')

In [ ]:
if entities.empty or rels.empty:
    print('No graph data — run: python -m graphrag index --root ragtest')
else:
    # Build graph from top entities by degree
    top_entities = entities.nlargest(40, 'degree') if 'degree' in entities.columns else entities.head(40)
    top_names = set(top_entities['title'].str.lower())

    G = nx.Graph()
    for _, row in top_entities.iterrows():
        G.add_node(row['title'], node_type=row.get('type', 'UNKNOWN'))

    for _, row in rels.iterrows():
        s, t = row['source'], row['target']
        if s.lower() in top_names and t.lower() in top_names:
            w = float(row.get('weight', 1))
            G.add_edge(s, t, weight=w)

    # Color by entity type
    type_color_map = {
        'PERSON': '#E74C3C', 'ORGANIZATION': '#3498DB', 'GEO': '#2ECC71',
        'EVENT': '#F39C12', 'CONCEPT': '#9B59B6', 'UNKNOWN': '#95A5A6'
    }
    node_colors = [type_color_map.get(G.nodes[n].get('node_type', 'UNKNOWN').upper(), '#95A5A6')
                   for n in G.nodes]
    node_sizes = [300 + 80 * G.degree(n) for n in G.nodes]

    fig, ax = plt.subplots(figsize=(18, 12))
    pos = nx.spring_layout(G, seed=42, k=2.5, iterations=60)

    nx.draw_networkx_edges(G, pos, alpha=0.3, width=0.8, edge_color='#aaa', ax=ax)
    nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=node_sizes, alpha=0.9, ax=ax)
    nx.draw_networkx_labels(G, pos, font_size=7, font_color='white', font_weight='bold', ax=ax)

    # Legend
    legend_handles = [mpatches.Patch(color=c, label=t) for t, c in type_color_map.items() if t != 'UNKNOWN']
    ax.legend(handles=legend_handles, loc='upper left', framealpha=0.9, fontsize=9)

    ax.set_title(f'GraphRAG Knowledge Graph — top {len(G.nodes)} entities by degree', fontsize=14, pad=15)
    ax.axis('off')
    plt.tight_layout()
    plt.savefig(ROOT / 'output' / 'viz_knowledge_graph.png', bbox_inches='tight', dpi=150)
    plt.show()
    print(f'Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges')

## 7. Community structure

In [ ]:
communities = load_parquet('communities')
reports = load_parquet('community_reports')

if not communities.empty and 'level' in communities.columns:
    level_counts = communities.groupby('level').size()

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # Communities per level
    axes[0].bar(level_counts.index.astype(str), level_counts.values, color='#9B59B6')
    axes[0].set_xlabel('Community level (0 = finest granularity)')
    axes[0].set_ylabel('Number of communities')
    axes[0].set_title('GraphRAG Community Hierarchy')

    # Community size distribution
    if 'size' in communities.columns:
        axes[1].hist(communities['size'], bins=15, color='#3498DB', edgecolor='white')
        axes[1].set_xlabel('Community size (entities)')
        axes[1].set_ylabel('Count')
        axes[1].set_title('Community Size Distribution')

    plt.tight_layout()
    plt.savefig(ROOT / 'output' / 'viz_communities.png', bbox_inches='tight')
    plt.show()

    print('\nTop community reports:')
    if not reports.empty:
        for _, r in reports.head(3).iterrows():
            print(f"\n  [{r.get('level','?')}] {r.get('title','—')}")
            print(f"  {r.get('summary','')[:300]}")
else:
    print('No community data — run indexing first.')

## 8. Standard RAG: retrieved chunks visualization

In [ ]:
# Retrieval source distribution across all queries
if data['standard_rag']:
    source_hits = {}
    score_data = []

    for r in data['standard_rag']:
        for chunk in r.get('retrieved_sources', []):
            src = chunk['source'].replace('.txt', '')
            source_hits[src] = source_hits.get(src, 0) + 1
            score_data.append({'question': r['id'], 'source': src, 'score': chunk['score']})

    score_df = pd.DataFrame(score_data)

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    # Source frequency
    ax = axes[0]
    srcs = list(source_hits.keys())
    counts = [source_hits[s] for s in srcs]
    ax.barh(srcs, counts, color='#4C72B0')
    ax.set_xlabel('Times retrieved across all queries')
    ax.set_title('Standard RAG: Source Retrieval Frequency')

    # Retrieval score heatmap (question × source)
    ax2 = axes[1]
    pivot = score_df.groupby(['question','source'])['score'].max().unstack(fill_value=0)
    sns.heatmap(pivot, annot=True, fmt='.2f', cmap='Blues', ax=ax2,
                cbar_kws={'label': 'Max cosine similarity'})
    ax2.set_title('Standard RAG: Max Retrieval Score\nper Question × Document')
    ax2.set_ylabel('')

    plt.tight_layout()
    plt.savefig(ROOT / 'output' / 'viz_rag_retrieval.png', bbox_inches='tight')
    plt.show()

## 9. Summary: key architectural differences

| | Standard RAG | GraphRAG |
|---|---|---|
| **Retrieval unit** | Text chunks (fixed window) | Entities + community reports |
| **Context** | Top-k similar passages | Graph neighbourhood + summaries |
| **Best for** | Factual lookups, simple Q&A | Relational, thematic, multi-hop |
| **Index cost** | Low (embeddings only) | High (many LLM extraction calls) |
| **Query latency** | Fast (embedding + 1 LLM call) | Moderate–slow (map-reduce over communities) |
| **Cross-doc reasoning** | Weak (only retrieves top-k chunks) | Strong (community summaries span docs) |
| **Hallucination risk** | Moderate | Lower (grounded in extracted facts) |

### When to use each

**Use Standard RAG when:**
- You need fast, cheap indexing
- Questions are mostly factual / single-document
- Your corpus is small or changes frequently

**Use GraphRAG when:**
- You need to reason about *relationships* between entities
- Questions require synthesising across many documents
- You want explainable, structured knowledge representation
- Corpus is large, stable, and entity-rich (reports, books, research papers)

In [ ]:
# Print all saved visualizations
output_dir = ROOT / 'output'
print('Saved visualizations:')
for f in sorted(output_dir.glob('viz_*.png')):
    print(f'  {f.name}')